In [2]:
def lambda_recogidaDatosFestivos():
#Es el lambda, aqui no se ejecuta
    import json
    import requests
    import boto3
    import time
    from datetime import datetime
    from requests.exceptions import RequestException
    
    s3 = boto3.client("s3")
    sqs = boto3.client("sqs")
    
    nombre_bucket = "angel-barrientos-simo-prada-big-data"
    
    queue_url = "https://sqs.us-east-1.amazonaws.com/891377024918/errores_proyecto"
    
    
    def enviar_error_sqs(ano, error, url):
    
        mensaje = {
            "ano": ano,
            "error": str(error),
            "url": url,
            "timestamp": datetime.utcnow().isoformat()
        }
    
        sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps(mensaje)
        )
    
    
    def lambda_handler(event, context):
    
        ano_actual = datetime.today().year
    
        start_year = event.get('start_year', 2013)
        end_year = event.get('end_year', ano_actual)
    
        resultados = []
    
        for ano in range(start_year, end_year + 1):
    
            api_key = "?api_key=KLPGqTnv9gejGEoxyYBGkHQ7S82mOIoy"
            pais = "&country=es"
    
            url = (
                f"https://calendarific.com/api/v2/holidays"
                f"{api_key}{pais}&year={ano}"
            )
    
            print(f"Procesando festivos del año: {ano}")
    
            try:
    
                response = requests.get(url, timeout=20)
    
                response.raise_for_status()
    
                datos = response.json()
    
                s3_key = f"bronce/calendario/year={ano}/festivos_{ano}.json"
    
                s3.put_object(
                    Bucket=nombre_bucket,
                    Key=s3_key,
                    Body=json.dumps(datos)
                )
    
                resultados.append(f"Éxito: {s3_key}")
    
            except RequestException as e:
    
                enviar_error_sqs(
                    ano,
                    f"Error HTTP: {str(e)}",
                    url
                )
    
                resultados.append(f"Error HTTP en {ano}")
    
            except Exception as e:
    
                enviar_error_sqs(
                    ano,
                    f"Error general: {str(e)}",
                    url
                )
    
                resultados.append(f"Error general en {ano}")
    
            time.sleep(1)
    
        return {
            'statusCode': 200,
            'body': json.dumps(resultados)
        }

In [3]:
def lambda_festivos():
#Este codigo es el del lambda aqui no funciona
    import requests
    import time
    import json
    import boto3
    from datetime import datetime
    from requests.exceptions import RequestException
    
    
    s3 = boto3.client("s3")
    sqs = boto3.client("sqs")
    
    
    nombre_bucket = "angel-barrientos-simo-prada-big-data"
    
    queue_url = "https://sqs.us-east-1.amazonaws.com/891377024918/errores_proyecto"
    
    
    def enviar_error_sqs(ano, mes, error, url):
    
        mensaje = {
            "ano": ano,
            "mes": mes,
            "error": str(error),
            "url": url,
            "timestamp": datetime.utcnow().isoformat()
        }
    
        sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps(mensaje)
        )
    
        print(f"Error enviado a SQS: {ano}-{mes}")
    
    
    def lambda_handler(event, context):
    
        ano_actual = datetime.today().year
    
        meses = [
            "01","02","03","04","05","06",
            "07","08","09","10","11","12"
        ]
    
        for ano in range(2013, ano_actual + 1):
    
            for mes in meses:
    
                datos_api_precios_mes = []
    
                url = (
                    f"https://apidatos.ree.es/es/datos/mercados/"
                    f"precios-mercados-tiempo-real?"
                    f"start_date={ano}-{mes}-01T00:00"
                    f"&end_date={ano}-{mes}-31T23:59"
                    f"&time_trunc=hour"
                )
    
                try:
    
                    response = requests.get(url, timeout=10)
    
                    
                    response.raise_for_status()
    
                    datos = response.json()
    
                    if "included" in datos:
    
                        for valor in datos["included"]:
    
                            datos_api_precios_mes.append(
                                valor["attributes"]["values"]
                            )
    
                    
                    s3_key = (
                        f"bronce/precio/year={ano}/"
                        f"month={mes}/precio_{ano}_{mes}.json"
                    )
    
                    s3.put_object(
                        Bucket=nombre_bucket,
                        Key=s3_key,
                        Body=json.dumps(datos_api_precios_mes)
                    )
    
                    print(f"Mes {mes} del año {ano} subido a: {s3_key}")
    
                except RequestException as e:
    
                    print(f"[ERROR HTTP] {ano}-{mes}: {str(e)}")
    
                    enviar_error_sqs(
                        ano,
                        mes,
                        f"Error HTTP: {str(e)}",
                        url
                    )
    
                except Exception as e:
    
                    print(f"[ERROR GENERAL] {ano}-{mes}: {str(e)}")
    
                    enviar_error_sqs(
                        ano,
                        mes,
                        f"Error general: {str(e)}",
                        url
                    )
    
            print(f"Datos del año {ano} completados")
    
            time.sleep(1)
    
        return {
            "statusCode": 200,
            "body": json.dumps(
                "Carga por carpetas finalizada correctamente"
            )
        }

In [4]:
def lambda_red_electrica():
#Este es un lambda, aqui no funciona
    import json
    import requests
    import boto3
    from datetime import datetime
    from requests.exceptions import RequestException
    
    s3 = boto3.client('s3')
    sqs = boto3.client("sqs")
    
    bucket_name = "angel-barrientos-simo-prada-big-data"
    
    queue_url = "https://sqs.us-east-1.amazonaws.com/891377024918/errores_proyecto"
    
    
    def enviar_error_sqs(ano, error, url):
    
        mensaje = {
            "ano": ano,
            "error": str(error),
            "url": url,
            "timestamp": datetime.utcnow().isoformat()
        }
    
        sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps(mensaje)
        )
    
    
    def lambda_handler(event, context):
    
        start_year = event.get('start_year', 2013)
    
        end_year = event.get(
            'end_year',
            datetime.today().year
        )
    
        resultados = []
    
        for ano in range(start_year, end_year + 1):
    
            url = (
                f"https://apidatos.ree.es/es/datos/"
                f"demanda/evolucion?"
                f"start_date={ano}-01-01T00:00"
                f"&end_date={ano}-12-31T23:59"
                f"&time_trunc=day"
            )
    
            print(f"Solicitando datos de ESIOS para el año: {ano}")
    
            try:
    
                response = requests.get(url, timeout=20)
    
                response.raise_for_status()
    
                datos = response.json()
    
                s3_key = f"bronce/demanda/demanda_{ano}.json"
    
                s3.put_object(
                    Bucket=bucket_name,
                    Key=s3_key,
                    Body=json.dumps(datos)
                )
    
                resultados.append(f"Éxito: {s3_key}")
    
            except RequestException as e:
    
                print(f"Error HTTP en año {ano}: {str(e)}")
    
                enviar_error_sqs(
                    ano,
                    f"Error HTTP: {str(e)}",
                    url
                )
    
                resultados.append(f"Error HTTP {ano}")
    
            except Exception as e:
    
                print(f"Error procesando el año {ano}: {str(e)}")
    
                enviar_error_sqs(
                    ano,
                    f"Error general: {str(e)}",
                    url
                )
    
                resultados.append(f"Error general {ano}")
    
        return {
            'statusCode': 200,
            'body': json.dumps(resultados)
        }

In [5]:
def lambda_clima():
#El lambda no se ejecuta aqui
    import json
    import requests
    import boto3
    import os
    import urllib.parse
    from bs4 import BeautifulSoup
    from datetime import datetime
    from requests.exceptions import RequestException
    
    s3 = boto3.client('s3')
    sqs = boto3.client("sqs")
    
    bucket_name = "angel-barrientos-simo-prada-big-data"
    queue_url = "https://sqs.us-east-1.amazonaws.com/891377024918/errores_proyecto"
    url_base = "https://datosclima.es/Aemet2013/DescargaDatos.html"
    
    
    def enviar_error_sqs(nombre_archivo, error, url):
        mensaje = {
            "archivo": nombre_archivo,
            "error": str(error),
            "url": url,
            "timestamp": datetime.utcnow().isoformat()
        }
    
        sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps(mensaje)
        )
    
    
    def lambda_handler(event, context):
    
        try:
            response = requests.get(url_base, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            enlaces = soup.find_all('a', href=True)
            elementos = []
            for e in enlaces:
                href = e['href']
                if href.lower().endswith('.rar'):
                    url_completa = urllib.parse.urljoin(url_base, href)
                    elementos.append(url_completa)
    
            resultados = []
    
            for url_archivo in elementos:
                nombre_archivo = url_archivo.split("/")[-1]
                ruta_temporal = f"/tmp/{nombre_archivo}"
                try:
                    print(f"Descargando {nombre_archivo}...")
                    r = requests.get(url_archivo, timeout=60)
                    r.raise_for_status()
                    with open(ruta_temporal, "wb") as f:
                        f.write(r.content)
                    s3_key = f"bronce/clima/{nombre_archivo}"
                    s3.upload_file(
                        ruta_temporal,
                        bucket_name,
                        s3_key
                    )
                    os.remove(ruta_temporal)
                    resultados.append(f"Subido: {s3_key}")
                except RequestException as e:
                    enviar_error_sqs(
                        nombre_archivo,
                        f"Error HTTP: {str(e)}",
                        url_archivo
                    )
    
                    resultados.append(f"Error HTTP: {nombre_archivo}")
                except Exception as e:
    
                    enviar_error_sqs(
                        nombre_archivo,
                        f"Error general: {str(e)}",
                        url_archivo
                    )
                    resultados.append(f"Error general: {nombre_archivo}")
    
            return {
                'statusCode': 200,
                'body': json.dumps(resultados)
            }
    
        except Exception as e:
    
            enviar_error_sqs(
                "proceso_general",
                str(e),
                url_base
            )
    
            return {
                'statusCode': 500,
                'body': str(e)
            }

In [1]:
def demandaPlata():

    import os
    import shutil
    import boto3    
    from pyspark.sql.functions import (col,explode,from_json,to_date,lit)
    from pyspark.sql.types import (StructType,StructField,StringType,ArrayType)
    from pyspark.sql import SparkSession

    from pyspark.sql.functions import (col,explode,lit,to_timestamp,substring,to_date,split)

    ruta_demanda = "/workspace/silver/demanda/"
    ruta_parquet = "/workspace/silver/demanda/DEMANDA_FINAL"
    bucket_name = "angel-barrientos-simo-prada-big-data"
    prefix_s3 = "bronce/demanda/"

    os.makedirs(ruta_demanda, exist_ok=True)
    for archivo in os.listdir(ruta_demanda):
        if archivo.endswith(".json"):
            os.remove(
                os.path.join(
                    ruta_demanda,
                    archivo
                )
            )

    s3 = boto3.client("s3")
    respuesta = s3.list_objects_v2(
        Bucket=bucket_name,
        Prefix=prefix_s3
    )

    print("Descargando archivos demanda")
    for obj in respuesta.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".json"):
            nombre_archivo = key.split("/")[-1]
            s3.download_file(bucket_name,key,os.path.join(ruta_demanda,nombre_archivo))

    spark = (
        SparkSession.builder
        .appName("silverDemanda")
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )
    try:
        print("Procesando archivos demanda")
        archivos_json = [
            file for file in os.listdir(ruta_demanda)
            if file.endswith(".json")
        ]
        dfs_procesados = []
        for nombre_archivo in archivos_json:
            ruta_completa = os.path.join(ruta_demanda,nombre_archivo)
            print(f"Procesando: {nombre_archivo}")
            df_temp = spark.read.json(ruta_completa)
            df_limpio = (df_temp.select(explode(col("included")).alias("datos"))
                .select(col("datos.attributes.title").alias("titulo"),explode(col("datos.attributes.values")).alias("v"))
                .select(col("titulo"),col("v.value").alias("megavatios"),col("v.datetime").alias("fecha_hora"),lit(nombre_archivo).alias("archivo_origen"))
            )
            dfs_procesados.append(df_limpio)
        if len(dfs_procesados) > 0:
            df_final = dfs_procesados[0]
            for i in range(1,len(dfs_procesados)):
                df_final = df_final.union(dfs_procesados[i])
            df_demanda_plata = (df_final.filter(col("titulo") == "Demanda")
                .withColumn("fecha_hora_limpia",to_timestamp(substring(col("fecha_hora"),1,19)))
                .withColumn("fecha_cruce",to_date(col("fecha_hora_limpia")))
                .withColumn("megavatios",col("megavatios").cast("float"))
                .withColumn("hora",split(col("fecha_hora_limpia").cast("string")," ")[1])
                .select("fecha_cruce",col("megavatios").alias("megavatios_dia")         
                )
            )
            print("Muestra demanda plata")
            df_demanda_plata.show(10,truncate=False)
            print("Schema")
            df_demanda_plata.printSchema()
            total = df_demanda_plata.count()
            print(f"Total registros: {total}")
            
            if os.path.exists(ruta_parquet):
                shutil.rmtree(ruta_parquet)
            if total > 0:
                print("Guardando parquet demanda")
                (
                    df_demanda_plata.coalesce(1).write.mode("overwrite").parquet(ruta_parquet)
                )
                print(f"Parquet guardado en: {ruta_parquet}")
                print(os.listdir(ruta_parquet))
            else:
                print("No hay datos válidos")
        else:
            print("No se encontraron JSON")

        print("Borrando JSON")
        for archivo in os.listdir(ruta_demanda):
            if archivo.endswith(".json"):
                os.remove(os.path.join(ruta_demanda,archivo))
        print("PROCESO demandaPlata FINALIZADO")

    except Exception as e:
        print(f"ERROR: {e}")
    finally:
        spark.stop()
#demandaPlata()

In [2]:
def preciosPlata():#esta
    import os
    import shutil
    import boto3
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (col,explode,from_json,to_timestamp,substring,to_date,split)
    from pyspark.sql.types import (StructType,StructField,StringType,DoubleType,ArrayType)

    ruta_precios = "/workspace/silver/precios/"
    ruta_parquet = "/workspace/silver/precios/PRECIOS_FINAL"
    bucket_name = "angel-barrientos-simo-prada-big-data"
    prefix_s3 = "bronce/precio/"
    os.makedirs(ruta_precios, exist_ok=True)

    for archivo in os.listdir(ruta_precios):
        if archivo.endswith(".json"):
            os.remove(os.path.join(ruta_precios, archivo))

    s3 = boto3.client("s3")
    respuesta = s3.list_objects_v2(Bucket=bucket_name,Prefix=prefix_s3)

    print("Descargando archivos")
    for obj in respuesta.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".json"):
            nombre_archivo = key.split("/")[-1]
            s3.download_file(bucket_name,key,os.path.join(ruta_precios,nombre_archivo))
    spark = (
        SparkSession.builder
        .appName("silverPrecios")
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )

    schema = StructType([
        StructField("value", DoubleType(), True),
        StructField("percentage", DoubleType(), True),
        StructField("datetime", StringType(), True)
    ])
    schema_json = ArrayType(
        ArrayType(schema)
    )
    try:
        print("Procesando archivos")
        df_raw = (spark.read.option("wholetext", "true").text(ruta_precios + "*.json"))
        df_parsed = df_raw.withColumn("data",from_json(col("value"),schema_json))

        df_final = (
            df_parsed.select(explode(col("data")).alias("nivel_1"))
            .select(explode(col("nivel_1")).alias("v"))
            .select(col("v.datetime").alias("fecha_hora"),col("v.value").alias("precio_mwh"),col("v.percentage").alias("porcentaje"))
            .filter(col("fecha_hora").isNotNull())
        )

        df_precios_plata = (
            df_final.withColumn("fecha_hora_limpia",to_timestamp(substring(col("fecha_hora"),1,19)))
            .withColumn("fecha_cruce",to_date(col("fecha_hora_limpia")))
            .withColumn("precio_mwh",col("precio_mwh").cast("float"))
            .withColumn("hora",split(col("fecha_hora_limpia").cast("string")," ")[1])
            .select("fecha_cruce","hora","precio_mwh")
        )
        
        print("Muestra de datos")
        
        df_precios_plata.show(10,truncate=False)
        print("Schema")
        df_precios_plata.printSchema()
        total = df_precios_plata.count()
        print(f"Total registros: {total}")
        
        if os.path.exists(ruta_parquet):
            shutil.rmtree(ruta_parquet)
            
        if total > 0:
            print("Guardando parquet")
            (
                df_precios_plata.coalesce(1).write.mode("overwrite").parquet(ruta_parquet)
            )
            print(f"Parquet guardado en: {ruta_parquet}")
            print("Archivos generados")
            print(os.listdir(ruta_parquet))
        else:
            print("No hay datos válidos")
        print("Borrando JSON")

        for archivo in os.listdir(ruta_precios):
            if archivo.endswith(".json"):
                os.remove(os.path.join(ruta_precios,archivo))
        print("PROCESO preciosPlata FINALIZADO")

    except Exception as e:
        print(f"ERROR: {e}")
    finally:
        spark.stop()

#preciosPlata()

In [3]:
def festivosPlata():
    import os
    import shutil
    import boto3
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (col,explode,from_json,to_date,lit)
    from pyspark.sql.types import (StructType,StructField,StringType,ArrayType)

    ruta_json = "/workspace/silver/festivos/json/"
    ruta_parquet = "/workspace/silver/festivos/FESTIVOS_FINAL"
    bucket_name = "angel-barrientos-simo-prada-big-data"
    prefix_s3 = "bronce/calendario/"
    os.makedirs(ruta_json, exist_ok=True)

    s3 = boto3.client("s3")
    respuesta = s3.list_objects_v2(Bucket=bucket_name,Prefix=prefix_s3)
    print("Descargando archivos")

    for obj in respuesta.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".json"):
            nombre_archivo = key.split("/")[-1]
            s3.download_file(bucket_name,key,os.path.join(ruta_json, nombre_archivo))
    spark = (
        SparkSession.builder
        .appName("silverFestivos")
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )

    holiday_schema = StructType([
        StructField("name", StringType(), True),
        StructField("type", ArrayType(StringType()),True),
        StructField("date",StructType([StructField("iso",StringType(),True)]),True)])
    schema_festivos = StructType([StructField("response",StructType([StructField("holidays",ArrayType(holiday_schema),True)]),True)])
    try:
        print("Procesando archivos")
        
        df_raw = (spark.read.option("wholetext", "true").text(ruta_json + "*.json"))
        df_parsed = df_raw.withColumn("json_data",from_json(col("value"),schema_festivos))
        df_festivos = (   
            df_parsed.select(explode(col("json_data.response.holidays")).alias("h"))        
            .select(col("h.name").alias("nombre_festivo"),col("h.date.iso").alias("fecha"),col("h.type").getItem(0).alias("tipo_festivo"))
            .filter(col("fecha").isNotNull())        )
        df_festivos_plata = (      
            df_festivos.filter((col("tipo_festivo") == "National holiday") | (col("tipo_festivo") == "Local holiday"))        
            .withColumn("fecha_cruce",to_date(col("fecha")))        
            .withColumn("es_festivo",lit(1))        
            .select("fecha_cruce","nombre_festivo","tipo_festivo","es_festivo"))
        
        print("Muestra de datos")
        
        df_festivos_plata.show(10,truncate=False)
    
        print("Schema")
        df_festivos_plata.printSchema()
        total = df_festivos_plata.count()
        print(f"Total registros: {total}")

        if os.path.exists(ruta_parquet):
            shutil.rmtree(ruta_parquet)

        if total > 0:
            print("Guardando parquet")
            (
                df_festivos_plata.coalesce(1).write.mode("overwrite").parquet(ruta_parquet)
            )
            print(f"Parquet guardado en: {ruta_parquet}")
            print("Archivos generados")
            print(os.listdir(ruta_parquet))
        else:
            print("No hay datos válidos")
        print("Borrando JSON")
        for archivo in os.listdir(ruta_json):
            if archivo.endswith(".json"):
                os.remove(os.path.join(ruta_json,archivo))
        print("PROCESO festivosPlata FINALIZADO")
        
    except Exception as e:
        print(f"ERROR: {e}")
    finally:
        ruta = "/workspace/silver/festivos/json"
        if os.path.exists(ruta):
            shutil.rmtree(ruta)
            print("Carpeta borrada")
        spark.stop()

#festivosPlata()

In [4]:
def climaPlata():
    import os
    import io
    import shutil
    import boto3
    import pandas as pd
    import rarfile
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (col,split,trim,lower,regexp_replace,create_map,lit,concat_ws,to_date)

    ruta_base = "/workspace/silver/clima/"
    ruta_xls = os.path.join(ruta_base,"temp_descomprimidos")
    ruta_parquet = os.path.join(ruta_base,"CLIMA_FINAL")
    bucket_name = "angel-barrientos-simo-prada-big-data"
    prefix_s3 = "bronce/clima/"
    os.makedirs(ruta_base,exist_ok=True)
    os.makedirs(ruta_xls,exist_ok=True)
    
    s3 = boto3.client("s3")
    respuesta = s3.list_objects_v2(Bucket=bucket_name,Prefix=prefix_s3)
    print("Descargando archivos clima")

    for obj in respuesta.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".rar"):
            nombre_archivo = key.split("/")[-1]
            ruta_destino = os.path.join(ruta_base,nombre_archivo)
            s3.download_file(bucket_name,key,ruta_destino)
            

    print("Descomprimiendo archivos RAR")
    archivos_rar = [f for f in os.listdir(ruta_base)if f.endswith(".rar")]

    for rar_nombre in archivos_rar:
        ruta_rar = os.path.join(ruta_base,rar_nombre)
        try:
            with rarfile.RarFile(ruta_rar) as rf:
                rf.extractall(ruta_xls)
            

        except Exception as e:
            print(f"Error descomprimiendo {rar_nombre}: {e}")

    spark = (
        SparkSession.builder
        .appName("silverClima")
        .master("local[*]")
        .config("spark.driver.memory","4g")
        .getOrCreate()
    )

    def procesar_xls(datos_binarios):
        ruta, contenido = datos_binarios
        try:
            pdf = pd.read_excel(io.BytesIO(contenido),engine="xlrd",header=None)
            columnas = ["Estacion","Provincia","TMax","TMin","TMed","Racha","VMax","P00_24","P00_06","P06_12","P12_18","P18_24"]
            df_datos = pdf.iloc[5:].copy()
            df_datos = df_datos.iloc[:,:len(columnas)]
            df_datos.columns = columnas
            df_datos["Fecha"] = str(pdf.iloc[2, 0])
            return (df_datos.dropna(subset=["Estacion"]).astype(str).to_dict("records"))
        except Exception:
            return []
    try:
        print("Procesando archivos XLS")

        archivos_xls = [os.path.join(r, f)
            for r, d, fs in os.walk(ruta_xls)
            for f in fs
            if f.lower().endswith(".xls")
        ]
        if not archivos_xls:
            print("No hay archivos XLS")
            return

        rutas_spark = ",".join([f"file://{os.path.abspath(f)}"for f in archivos_xls])
        df_clima_raw = (
            spark.sparkContext
            .binaryFiles(rutas_spark)
            .flatMap(procesar_xls)
            .toDF()
        )
        print("Datos cargados")
        df_verificacion = (df_clima_raw
            .withColumn("Fecha_Temp",split(col("Fecha"),":")[1])
            .withColumn("Fecha_Temp",trim(col("Fecha_Temp")))
            .withColumn("Dia_Semana",split(col("Fecha_Temp"),",")[0])
            .withColumn("Resto_Fecha",trim(split(col("Fecha_Temp"),",")[1]))
            .withColumn("Dia_Num",split(col("Resto_Fecha")," ")[0])
            .withColumn("Mes_Nombre",lower(split(col("Resto_Fecha")," ")[1]))
            .withColumn("Anio",split(col("Resto_Fecha")," ")[2])
            .withColumn("TMax_Hora_aux",col("TMax"))
            .withColumn("TMax",split(col("TMax")," ")[0])
            .withColumn("TMax_Hora",split(col("TMax_Hora_aux")," ")[1])
            .withColumn("TMax_Hora",regexp_replace(col("TMax_Hora"),r"\(",""))
            .withColumn("TMax_Hora",regexp_replace(col("TMax_Hora"),r"\)",""))
            .withColumn("TMin_Hora_aux",col("TMin"))
            .withColumn("TMin",split(col("TMin")," ")[0])
            .withColumn("TMin_Hora",split(col("TMin_Hora_aux")," ")[1])
            .withColumn("TMin_Hora",regexp_replace(col("TMin_Hora"),r"\(",""))
            .withColumn("TMin_Hora",regexp_replace(col("TMin_Hora"),r"\)",""))
            .withColumn("VMax_aux",col("VMax"))
            .withColumn("VMax",split(col("VMax")," ")[0])
            .withColumn("VMax_Hora",split(col("VMax_aux")," ")[1])
            .withColumn("VMax_Hora",regexp_replace(col("VMax_Hora"),r"\(",""))
            .withColumn("VMax_Hora",regexp_replace(col("VMax_Hora"),r"\)",""))
            .withColumn("Racha_aux",col("Racha"))
            .withColumn("Racha",split(col("Racha")," ")[0])
            .withColumn("Racha_Hora",split(col("Racha_aux")," ")[1])
            .withColumn("Racha_Hora",regexp_replace(col("Racha_Hora"),r"\(",""))
            .withColumn("Racha_Hora",regexp_replace(col("Racha_Hora"),r"\)","")))

        meses_dict = {
            "enero": "01",
            "febrero": "02",
            "marzo": "03",
            "abril": "04",
            "mayo": "05",
            "junio": "06",
            "julio": "07",
            "agosto": "08",
            "septiembre": "09",
            "octubre": "10",
            "noviembre": "11",
            "diciembre": "12"
        }

        mapping_expr = create_map(
            [
                lit(x)
                for x in [ val for pair in meses_dict.items()for val in pair]
            ]
        )

        df_verificacion = (
            df_verificacion
            .withColumn("Mes_Num",mapping_expr.getItem(col("Mes_Nombre")))
            .withColumn("Fecha_Limpia",concat_ws("-",col("Anio"),col("Mes_Num"),col("Dia_Num"))))

        df_clima_plata = (
            df_verificacion
            .select("Estacion","Provincia",to_date(col("Fecha_Limpia")).alias("fecha_cruce"),"Dia_Semana",col("P00_06").cast("float"),col("P00_24").cast("float"),col("P06_12").cast("float"),col("P12_18").cast("float"),col("P18_24").cast("float"),col("Racha").cast("float"),"Racha_Hora",col("TMax").cast("float"),"TMax_Hora",col("TMed").cast("float"),col("TMin").cast("float"),"TMin_Hora",col("VMax").cast("float"),"VMax_Hora"))
        
        print("Muestra de datos")
        df_clima_plata.show(10,truncate=False)
        print("Schema")
        df_clima_plata.printSchema()
        total = df_clima_plata.count()
        print(f"Total registros: {total}")

        if os.path.exists(ruta_parquet):
            shutil.rmtree(ruta_parquet)
    
        if total > 0:
            print("Guardando parquet")
            (
                df_clima_plata.coalesce(1).write.mode("overwrite").parquet(ruta_parquet)
            )
            print(f"Parquet guardado en: {ruta_parquet}")
            print(os.listdir(ruta_parquet))
        else:
            print("No hay datos válidos")

        print("Borrando archivos temporales")

        for archivo in os.listdir(ruta_base):
            if archivo.endswith(".rar"):
                try:
                    os.remove(os.path.join(ruta_base,archivo))
                except Exception as e:
                    print(f"Error borrando RAR {archivo}: {e}")

        if os.path.exists(ruta_xls):
            try:
                shutil.rmtree(ruta_xls)
                print("Carpeta temp_descomprimidos borrada")
            except Exception as e:
                print(f"Error borrando XLS: {e}")
        print("PROCESO climaPlata FINALIZADO")
    except Exception as e:
        print(f"ERROR: {e}")
    finally:
        spark.stop()
#climaPlata()

In [5]:
def unionDatos():

    import os
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (col,when,avg,month,year,dayofweek,isnan)
    listaRutas = ["precios/PRECIOS_FINAL","demanda/DEMANDA_FINAL","festivos/FESTIVOS_FINAL","clima/CLIMA_FINAL"]
    contador = 0
    for rutas in listaRutas:
        for archivo in os.listdir(f"/workspace/silver/{rutas}"):
            if archivo.lower().endswith(".parquet"):
                rutaCompleta = f"/workspace/silver/{rutas}/{archivo}"
                if contador == 0:
                    rutaPartPrecio = rutaCompleta
                elif contador == 1:
                    rutaPartDemanda = rutaCompleta
                elif contador == 2:
                    rutaPartFestivos = rutaCompleta
                elif contador == 3:
                    rutaPartClima = rutaCompleta
                contador += 1
    spark = (
        SparkSession.builder
        .appName("goldEnergia")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )

    df_precios = spark.read.parquet(rutaPartPrecio)
    df_demanda = spark.read.parquet(rutaPartDemanda)
    df_festivos = spark.read.parquet(rutaPartFestivos)
    df_clima = spark.read.parquet(rutaPartClima)
    print("\nDATOS LEÍDOS")

    columnas_clima = ["TMed","TMax","TMin","Racha","P00_24"]
    for c in columnas_clima:
        df_clima = (
            df_clima.withColumn(c,when(isnan(col(c)),None).otherwise(col(c)))
        )

    df_clima_agg = (
        df_clima.groupBy("fecha_cruce")
        .agg(
            avg("TMed").alias("temp_media"),
            avg("TMax").alias("temp_max_media"),
            avg("TMin").alias("temp_min_media"),
            avg("Racha").alias("racha_media"),
            avg("P00_24").alias("precipitacion_media")
        )
    )

    df_precios_agg = (
        df_precios
        .groupBy("fecha_cruce")
        .agg(
            avg("precio_mwh").alias("precio_medio_dia")
        )
    )


    df_festivos = (
        df_festivos.dropDuplicates(["fecha_cruce"])
    )


    df_gold = (df_demanda.join(df_festivos,["fecha_cruce"],"left"))


    df_gold = (df_gold.join(df_clima_agg,["fecha_cruce"],"left"))

    df_gold = (df_gold.join(df_precios_agg,["fecha_cruce"],"left"))


    df_gold = (df_gold.fillna({
        "es_festivo": 0,
        "nombre_festivo": "No es festivo",
        "tipo_festivo": "No es festivo"
    })
        .withColumn("anio",year(col("fecha_cruce")))
        .withColumn("mes",month(col("fecha_cruce")))
        .withColumn("fin_de_semana",when(dayofweek(col("fecha_cruce")).isin([1, 7]),1).otherwise(0))
        .withColumn("rango_temperatura",col("temp_max_media") - col("temp_min_media"))
        .withColumn("precio_alto",when(col("precio_medio_dia") > 100,1).otherwise(0))
    )
    total = df_gold.count()
    print(f"\nTOTAL FILAS GOLD: {total}")

    ruta_gold = "/workspace/gold/ENERGIA_GOLD"
    df_gold = df_gold.select(col("fecha_cruce"),col("es_festivo"),col("temp_media"),col("temp_max_media"),col("temp_min_media"),col("racha_media"),col("precipitacion_media"),col("precio_medio_dia"),col("anio"),col("mes"),col("fin_de_semana"),col("rango_temperatura"),col("precio_alto"),col("megavatios_dia").alias("VAR_OBJETIVO_megavatios_dia"))
    from pyspark.sql.functions import col, when, sum as spark_sum

    
    columnas = df_gold.columns
    max_nulls = int(len(columnas) * 0.4)

    expr_nulls = when(col(columnas[0]).isNull(), 1).otherwise(0)

    for c in columnas[1:]:
        expr_nulls = expr_nulls + when(col(c).isNull(), 1).otherwise(0)

    df_gold = (df_gold
        .withColumn("num_nulls", expr_nulls).filter(col("num_nulls") <= max_nulls).drop("num_nulls")
    )
    (
        df_gold.coalesce(1).write.mode("overwrite").parquet(ruta_gold)
    )
    print(f"\nGOLD GUARDADO EN: {ruta_gold}")
    print("\nARCHIVOS GOLD")
    print(os.listdir(ruta_gold))
    spark.stop()
    print("\nPROCESO unionDatos FINALIZADO")
# unionDatos()

In [6]:
def muestreoDatos():
    import os
    from datetime import datetime
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (col,split,trim,lower,regexp_replace,create_map,lit,concat_ws,to_date)
    anio_actual = datetime.now().year
    spark = (
            SparkSession.builder
            .appName("Muestreo")
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .getOrCreate()
        )
    for archivo in os.listdir(f"/workspace/gold/ENERGIA_GOLD"):
            if archivo.lower().endswith(".parquet"):
                ruta = f"/workspace/gold/ENERGIA_GOLD/{archivo}"

    df_gold = spark.read.parquet(ruta)
    total = df_gold.count()
    print(total)
    for valor in range(2013,anio_actual + 1):
        df_gold.filter(
            (col("fecha_cruce") >= f"{valor}-01-01") &
            (col("fecha_cruce") <= f"{valor}-12-31")
        ).orderBy("fecha_cruce").show(30, False)

In [7]:
def generar_dashboard():
    from pyspark.sql import SparkSession
    import pandas as pd
    import altair as alt
    import os
    alt.data_transformers.disable_max_rows()
    
    ANCHO_GRANDE = 1100
    ANCHO_MEDIO = 540
    ALTO_GRANDE = 350
    ALTO_MEDIO = 320
    
    spark = (
        SparkSession.builder
        .appName("Dashboard Energia")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )
    for archivo in os.listdir(f"/workspace/gold/ENERGIA_GOLD"):
            if archivo.lower().endswith(".parquet"):
                ruta = f"/workspace/gold/ENERGIA_GOLD/{archivo}"

    df_spark = spark.read.parquet(ruta)
    df = df_spark.toPandas()
    df["fecha_cruce"] = pd.to_datetime(df["fecha_cruce"])
    df["dia_semana"] = df["fecha_cruce"].dt.day_name()
    df["mes_nombre"] = df["fecha_cruce"].dt.month_name()
    df["tipo_dia"] = df["es_festivo"].map({0: "Laboral",1: "Festivo"})
    
    float_cols = ["temp_media","temp_max_media","temp_min_media","racha_media","precipitacion_media","precio_medio_dia","VAR_OBJETIVO_megavatios_dia"]
    
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    
    print("\nKPIs PRINCIPALES")
    
    print(f"Demanda media: {df['VAR_OBJETIVO_megavatios_dia'].mean():,.2f} MW")
    print(f"Pico máximo: {df['VAR_OBJETIVO_megavatios_dia'].max():,.2f} MW")
    print(f"Temperatura media: {df['temp_media'].mean():,.2f} ºC")
    print(f"Precio medio electricidad:{df['precio_medio_dia'].mean():,.2f} €")
    
    grafico_demanda = (
        alt.Chart(df)
        .mark_line()
        .encode(
            x=alt.X("fecha_cruce:T",title="Fecha"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda MW"),
            tooltip=["fecha_cruce:T","VAR_OBJETIVO_megavatios_dia:Q"]
        )
        .properties(
            title="Evolución Temporal de la Demanda Eléctrica",
            width=ANCHO_GRANDE,
            height=ALTO_GRANDE
        )
        .interactive()
    )
    
    dias_orden = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    
    demanda_dia = (
        df.groupby("dia_semana")["VAR_OBJETIVO_megavatios_dia"]
        .mean()
        .reset_index()
    )
    
    demanda_dia["dia_semana"] = pd.Categorical(
        demanda_dia["dia_semana"],
        categories=dias_orden,
        ordered=True
    )
    
    demanda_dia = demanda_dia.sort_values("dia_semana")
    
    grafico_dias = (
        alt.Chart(demanda_dia)
        .mark_bar()
        .encode(
            x=alt.X("dia_semana:N",title="Día Semana"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda Media"),
            tooltip=["dia_semana","VAR_OBJETIVO_megavatios_dia"]
        )
        .properties(
            title="Demanda Media por Día de la Semana",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    grafico_temp = (
        alt.Chart(df)
        .mark_circle(size=70)
        .encode(
            x=alt.X("temp_media:Q",title="Temperatura Media"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda"),
            tooltip=["temp_media","VAR_OBJETIVO_megavatios_dia","fecha_cruce"]
        )
        .properties(
            title="Relación entre Temperatura y Demanda",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
        .interactive()
    )
    
    grafico_precio = (
        alt.Chart(df)
        .mark_circle(size=70)
        .encode(
            x=alt.X("precio_medio_dia:Q",title="Precio Medio Electricidad"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda"),
            tooltip=["precio_medio_dia","VAR_OBJETIVO_megavatios_dia"]
        )
        .properties(
            title="Precio Eléctrico vs Demanda",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
        .interactive()
    )
    
    grafico_boxplot = (
        alt.Chart(df)
        .mark_boxplot()
        .encode(
            x=alt.X("tipo_dia:N",title="Tipo Día"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda")
        )
        .properties(
            title="Comparación Laboral vs Festivo",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    grafico_heatmap = (
        alt.Chart(df)
        .mark_rect()
        .encode(
            x=alt.X("mes:O",title="Mes"),
            y=alt.Y("anio:O",title="Año"),
            color=alt.Color("mean(VAR_OBJETIVO_megavatios_dia):Q",title="Demanda Media"),
            tooltip=["mes","anio",alt.Tooltip("mean(VAR_OBJETIVO_megavatios_dia):Q",title="Demanda Media")]
        )
        .properties(
            title="Heatmap de Demanda Media",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    corr_cols = ["temp_media","temp_max_media","temp_min_media","racha_media","precipitacion_media","precio_medio_dia","VAR_OBJETIVO_megavatios_dia"]
    
    corr = df[corr_cols].corr()
    corr_objetivo = (corr["VAR_OBJETIVO_megavatios_dia"].drop("VAR_OBJETIVO_megavatios_dia").reset_index())
    
    corr_objetivo.columns = ["variable","correlacion"]
    
    grafico_corr_objetivo = (
        alt.Chart(corr_objetivo)
        .mark_bar()
        .encode(
            x=alt.X("correlacion:Q",title="Correlación"),
            y=alt.Y("variable:N",sort="-x",title="Variable"),
            color=alt.Color("correlacion:Q",scale=alt.Scale(scheme="redblue"),title="Correlación"),
            tooltip=[alt.Tooltip("variable:N",title="Variable"),alt.Tooltip("correlacion:Q",title="Correlación",format=".3f")]
        )
        .properties(
            title="Correlación con la Demanda Eléctrica",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    grafico_temp_box = (
        alt.Chart(df)
        .mark_boxplot()
        .encode(
            x=alt.X("rango_temperatura:Q",title="Rango Temperatura"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda")
        )
        .properties(
            title="Demanda según Temperatura",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    grafico_mes = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("mes:O",title="Mes"),
            y=alt.Y("mean(VAR_OBJETIVO_megavatios_dia):Q",title="Demanda Media"),
            tooltip=["mes",alt.Tooltip("mean(VAR_OBJETIVO_megavatios_dia):Q",title="Demanda Media")]
        )
        .properties(
            title="Demanda Media por Mes",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    brush = alt.selection_interval()
    
    scatter = (
        alt.Chart(df)
        .mark_circle(size=70)
        .encode(
            x=alt.X("temp_media:Q",title="Temperatura"),
            y=alt.Y("VAR_OBJETIVO_megavatios_dia:Q",title="Demanda"),
            color=alt.condition(
                brush,
                alt.value("steelblue"),
                alt.value("lightgray")
            ),
            tooltip=["temp_media","VAR_OBJETIVO_megavatios_dia"]
        )
        .add_params(brush)
        .properties(
            title="Selección Interactiva",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    barras_filtradas = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("mes:O",title="Mes"),
            y=alt.Y("mean(VAR_OBJETIVO_megavatios_dia):Q",title="Demanda Media")
        )
        .transform_filter(brush)
        .properties(
            title="Datos Filtrados",
            width=ANCHO_MEDIO,
            height=ALTO_MEDIO
        )
    )
    
    dashboard = alt.vconcat(
    
        grafico_demanda,
        alt.hconcat(grafico_dias,grafico_boxplot,spacing=20),
        alt.hconcat(grafico_temp,grafico_precio,spacing=20),
        alt.hconcat(grafico_heatmap,grafico_mes,spacing=20),
        alt.hconcat(grafico_corr_objetivo,grafico_temp_box,spacing=20),
        alt.hconcat(scatter,barras_filtradas,spacing=20),
        spacing=35
    ).configure_view(
        strokeWidth=1,
        stroke="lightgray"
    )
    return dashboard

In [9]:
import time
def CapaOro():
    demandaPlata()
    time.sleep(10)
    preciosPlata()
    time.sleep(10)
    festivosPlata()
    time.sleep(10)
    climaPlata()
    print("Todos los datos han sido recogidos y tratados de manera individual")
    time.sleep(10)
    unionDatos()
    print("Muestreo de datos")
    muestreoDatos()
    generar_dashboard()
    print("Datos unidos en la capa oro \nProceso finalizado")

In [10]:
CapaOro()

Descargando archivos demanda


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 19:55:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Procesando archivos demanda
Procesando: demanda_2013.json


Procesando: demanda_2014.json
Procesando: demanda_2015.json
Procesando: demanda_2016.json
Procesando: demanda_2017.json
Procesando: demanda_2018.json
Procesando: demanda_2019.json
Procesando: demanda_2020.json
Procesando: demanda_2021.json
Procesando: demanda_2022.json
Procesando: demanda_2023.json
Procesando: demanda_2024.json
Procesando: demanda_2025.json
Procesando: demanda_2026.json
Muestra demanda plata


26/05/12 19:55:41 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


...

Parquet guardado en: /workspace/silver/clima/CLIMA_FINAL
['.part-00000-32c75987-81d5-4078-a475-b7125da7a3d5-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-32c75987-81d5-4078-a475-b7125da7a3d5-c000.snappy.parquet', '_SUCCESS']
Borrando archivos temporales
Carpeta temp_descomprimidos borrada
PROCESO climaPlata FINALIZADO
Todos los datos han sido recogidos y tratados de manera individual

DATOS LEÍDOS

TOTAL FILAS GOLD: 4879



GOLD GUARDADO EN: /workspace/gold/ENERGIA_GOLD

ARCHIVOS GOLD
['.part-00000-1e397f44-8982-407d-87bd-970b3e34cdfc-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-1e397f44-8982-407d-87bd-970b3e34cdfc-c000.snappy.parquet', '_SUCCESS']

PROCESO unionDatos FINALIZADO
Muestreo de datos
4574
+-----------+----------+------------------+------------------+-------------------+------------------+---------------------+----------------+----+---+-------------+------------------+-----------+---------------------------+
|fecha_cruce|es_festivo|temp_media        |temp_max_media    |temp_min_media     |racha_media       |precipitacion_media  |precio_medio_dia|anio|mes|fin_de_semana|rango_temperatura |precio_alto|VAR_OBJETIVO_megavatios_dia|
+-----------+----------+------------------+------------------+-------------------+------------------+---------------------+----------------+----+---+-------------+------------------+-----------+---------------------------+
|2013-05-07 |0         |18.3406294207

26/05/12 20:10:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.



KPIs PRINCIPALES
Demanda media: 706,021.38 MW
Pico máximo: 888,001.94 MW
Temperatura media: 15.71 ºC
Precio medio electricidad:82.49 €
Datos unidos en la capa oro 
Proceso finalizado


/tmp/ipykernel_26/2893931734.py:15: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  generar_dashboard()
/tmp/ipykernel_26/2893931734.py:15: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  generar_dashboard()


In [11]:
generar_dashboard()


KPIs PRINCIPALES
Demanda media: 706,021.38 MW
Pico máximo: 888,001.94 MW
Temperatura media: 15.71 ºC
Precio medio electricidad:82.49 €


/tmp/ipykernel_26/3794562748.py:1: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  generar_dashboard()
/tmp/ipykernel_26/3794562748.py:1: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  generar_dashboard()


alt.VConcatChart(...)